# Lecture 7 — Final Matrix Completion

In the previous notebook, we compared several candidate latent ranks and selected the best value of $k$. The next step is to use that selected rank in a final matrix-factorization model.

This notebook demonstrates the workflow on the same synthetic setting:

```text
select k  →  fit the final model  →  reconstruct X  →  predict missing entries
```

The complete synthetic matrix is retained only so that we can visualize the predictions. In a real recommender system, genuinely missing ratings are unknown, so their true values cannot be used to calculate RMSE.

## 1. Generate the synthetic rating matrix

We generate a complete low-rank matrix $X_{\text{true}}$ and hide some of its entries. The factorization algorithm receives only the observed entries.

For this experiment, notebook 02 selected

$$
k^*=2.
$$

The hidden entries are used here only to illustrate what the model is trying to predict.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

In [ ]:
def objective_rank_k(Y, U, V, lam):
    mask = ~np.isnan(Y)
    residuals = Y[mask] - (U @ V.T)[mask]
    return 0.5 * np.sum(residuals**2) + 0.5 * lam * (np.sum(U**2) + np.sum(V**2))

def factor_rank_k(Y, k, lam=0.05, max_iter=500, tol=1e-8, seed=1):
    rng = np.random.default_rng(seed)
    n, m = Y.shape
    U = 0.1 * rng.normal(size=(n, k))
    V = 0.1 * rng.normal(size=(m, k))
    I = np.eye(k)
    history = []

    for _ in range(max_iter):
        for a in range(n):
            observed = ~np.isnan(Y[a])
            A = V[observed].T @ V[observed] + lam * I
            b = V[observed].T @ Y[a, observed]
            U[a] = np.linalg.solve(A, b)

        for i in range(m):
            observed = ~np.isnan(Y[:, i])
            A = U[observed].T @ U[observed] + lam * I
            b = U[observed].T @ Y[observed, i]
            V[i] = np.linalg.solve(A, b)

        history.append(objective_rank_k(Y, U, V, lam))
        if len(history) > 1 and abs(history[-1] - history[-2]) < tol:
            break

    return U, V, np.array(history)

In [ ]:
rng = np.random.default_rng(42)
n_users, n_movies, k_true = 8, 7, 2
U_true = rng.normal(size=(n_users, k_true))
V_true = rng.normal(size=(n_movies, k_true))
X_true = U_true @ V_true.T

mask = rng.random((n_users, n_movies)) < 0.55
Y_synth = np.where(mask, X_true, np.nan)

print('Observed matrix Y:')
print(Y_synth)

## 2. Fit the final model with the selected rank

Now that $k^*=2$ has been selected, we no longer compare different ranks. We fit one final model using that rank and **all ratings available for training**.

$$
U^*,V^*=\operatorname{Factor}(Y,k^*).
$$

In this synthetic example, `Y_synth` contains all the ratings designated as observed. In a real experiment, this would correspond to fitting the final model after model selection using the available training and validation ratings.

In [ ]:
best_k = 2
U_final, V_final, history = factor_rank_k(
    Y_synth,
    k=best_k,
    lam=0.05,
    seed=1,
)

X_hat = U_final @ V_final.T

print(f'Selected k: {best_k}')
print('U_final:')
print(U_final)
print('\nV_final:')
print(V_final)
print('\nX_hat:')
print(X_hat)

## 3. Predict the missing entries

The reconstructed matrix $\hat X=U^*{V^*}^T$ contains a prediction for every user-movie pair.

For an observed entry, we already have a real rating. For a missing entry, the corresponding value of $\hat X$ is the model's prediction.

We therefore keep observed ratings unchanged and fill only the missing positions from $\hat X$.

In [ ]:
X_completed = Y_synth.copy()
X_completed[~mask] = X_hat[~mask]

print('Completed matrix:')
print(X_completed)

## 4. Compare with the synthetic truth

Because this is a synthetic experiment, we can inspect the hidden entries alongside their true values. This is useful for understanding the experiment, but it is **not available for genuinely missing ratings in a real recommender system**.

The comparison below shows only the entries that were hidden from the model.

In [ ]:
hidden_indices = np.argwhere(~mask)

print('Hidden entries:')
print('user  movie  true  predicted')
for a, i in hidden_indices:
    print(f'{a:4d}  {i:5d}  {X_true[a, i]:5.3f}  {X_hat[a, i]:9.3f}')

## 5. From the experiment to a real recommender system

The synthetic experiment makes the complete workflow visible:

1. Start with observed ratings.
2. Split known ratings into training, validation, and test sets when evaluating a real system.
3. Train candidate models for several values of $k$.
4. Select $k^*$ using validation performance.
5. Refit the final model with the selected $k^*$ using all data available for final training.
6. Compute $\hat X=U^*{V^*}^T$.
7. Use $\hat X_{ai}$ for user-movie pairs whose ratings are genuinely missing.
8. Use the untouched test set only for the final evaluation of the selected modeling procedure.

The key distinction is that validation/test ratings are temporarily hidden **known ratings**, while genuinely missing ratings have no known target value.

## What to remember

- $k$ is selected as a hyperparameter; it is not learned as an entry of $U$ or $V$.
- Once $k^*$ is selected, fit the final factorization with that rank.
- $U^*$ and $V^*$ are the learned latent-factor matrices of the final model.
- $\hat X=U^*{V^*}^T$ gives a prediction for every user-movie pair.
- Keep observed ratings and use predictions for genuinely missing entries.
- In the real world, the quality of genuinely missing predictions cannot be measured directly because their true ratings are unknown.